# 02.5 - Dlib Method D (Letterboxed 96×96) Lip ROI Cropping & 100-Speaker Augmented Dataset Generation
## Tai Phake Visual Speech Recognition (VSR) Project

**Objective**: Construct an end-to-end dataset generation pipeline that produces a high-quality, letterboxed $96 \times 96$ lip-cropped dataset using **Dlib 68-landmark mouth geometry (Method D)**, expanded via **conservative, lip-reading-safe data augmentation** into exactly **100 speaker directories**:
- **Train Split**: Exactly **80 folders** derived from **24 base speakers** (`s1` through `s24`)
- **Validation Split**: Exactly **20 folders** derived from **6 base speakers** (`s25` through `s30`)
- **Zero Data Leakage**: The 24 train base speakers and 6 validation base speakers are strictly disjoint ($Train \cap Val = \emptyset$). No base speaker or variant crosses split boundaries.

---

### Core Methodology & Specifications:

1. **Method D Letterboxed Cropping ($96 \times 96$)**:
   - Extracts mouth landmarks (points 48–67) using Dlib 68 facial landmark detector.
   - Localizes tight mouth bounding box with standard padding ($P_x = 15\text{ px}, P_y = 10\text{ px}$) around mouth centroid $(c_x, c_y)$.
   - Performs **proportional, aspect-preserving scaling**:
     $$\text{scale} = \frac{96}{\max(W_{\text{roi}}, H_{\text{roi}})}$$
     $$W_{\text{scaled}} = \lfloor W_{\text{roi}} \cdot \text{scale} \rceil, \quad H_{\text{scaled}} = \lfloor H_{\text{roi}} \cdot \text{scale} \rceil$$
   - Centers the scaled mouth ROI onto a neutral black $96 \times 96$ canvas (letterboxing).
   - **Key Advantage**: Preserves natural ~2:1 lip aspect ratio without vertical distortion (Method A), avoids excessive facial background noise (Method B), and stays 100% compliant with standard $96 \times 96$ neural network inputs (unlike non-square Method C).

2. **Deterministic 80/20 Base-Speaker Expansion (100 Folders Total)**:
   - **Total Base Speakers**: 30 canonical speakers (`s1` to `s30`).
   - **Train Base Speakers (24)**: `s1` through `s24`.
     - 8 speakers (`s1`..`s8`) receive 4 folders: 1 original (`sX`) + 3 augmentations (`sX_aug1`, `sX_aug2`, `sX_aug3`) $\to 8 \times 4 = 32$ folders.
     - 16 speakers (`s9`..`s24`) receive 3 folders: 1 original (`sX`) + 2 augmentations (`sX_aug1`, `sX_aug2`) $\to 16 \times 3 = 48$ folders.
     - **Total Train Folders**: $32 + 48 = \mathbf{80}$ folders.
   - **Validation Base Speakers (6)**: `s25` through `s30`.
     - 2 speakers (`s25`..`s26`) receive 4 folders: 1 original (`sX`) + 3 augmentations (`sX_aug1`, `sX_aug2`, `sX_aug3`) $\to 2 \times 4 = 8$ folders.
     - 4 speakers (`s27`..`s30`) receive 3 folders: 1 original (`sX`) + 2 augmentations (`sX_aug1`, `sX_aug2`) $\to 4 \times 3 = 12$ folders.
     - **Total Validation Folders**: $8 + 12 = \mathbf{20}$ folders.
   - **Total Folders**: $80 + 20 = \mathbf{100}$ folders.

3. **Lip-Reading Safe Augmentations**:
   - **Temporally-Consistent Geometric Transforms**: Small translation ($\pm 3\text{ px}$), small rotation ($\pm 3^\circ$), and scale ($0.97 - 1.03$) sampled **once per utterance variant** to prevent inter-frame jitter.
   - **Photometric Variations**: Contrast ($0.92 - 1.08$), brightness ($\pm 10$), mild Gaussian noise (50% prob), and mild Gaussian blur (30% prob).
   - **Border Mode**: `cv2.BORDER_REFLECT_101` to prevent unnatural black seam artifacts at crop margins.

4. **Directory Structure**:
   Follows standard repo hierarchy under `data/`:
   ```
   data/experiments/dlib_augmented_cropped_100_96x96/
   ├── train/               (80 speaker folders)
   │   ├── s1/ .. s24/
   │   ├── s1_aug1/ ..
   │   └── ...
   │       └── d0/ .. d10/
   │           └── frame_0001.jpg ..
   ├── val/                 (20 speaker folders)
   │   ├── s25/ .. s30/
   │   ├── s25_aug1/ ..
   │   └── ...
   │       └── d0/ .. d10/
   │           └── frame_0001.jpg ..
   ├── augmentation_manifest.csv
   ├── speaker_split.json
   └── augmentation_summary.json
   ```

## 1. Environment, Imports & Centralized Configuration
Initialize core libraries, set deterministic random seeds, and configure dataset paths.

In [ ]:
import os
import sys
import re
import json
import time
import shutil
import hashlib
import random
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
from typing import Dict, List, Tuple, Optional, Any

import cv2
import dlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Reproducibility seed
DEFAULT_SEED = 42
random.seed(DEFAULT_SEED)
np.random.seed(DEFAULT_SEED)

# Paths Configuration
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
DATA_ROOT = PROJECT_ROOT / "data"

# Input paths
RAW_DIGIT_DIR = DATA_ROOT / "digit"
DLIB_PREDICTOR_PATH = PROJECT_ROOT / "models" / "dlib" / "shape_predictor_68_face_landmarks.dat"
GEOMETRY_CSV_PATH = PROJECT_ROOT / "results" / "dlib" / "dlib_roi_geometry_all_frames.csv"
PRE_CROPPED_METHOD_D_DIR = DATA_ROOT / "experiments" / "cropped_lips_dataset_d_96x96"

# Output dataset directory
DATASET_NAME = "dlib_augmented_cropped_100_96x96"
OUTPUT_DATASET_DIR = DATA_ROOT / "experiments" / DATASET_NAME
OUTPUT_TRAIN_DIR = OUTPUT_DATASET_DIR / "train"
OUTPUT_VAL_DIR = OUTPUT_DATASET_DIR / "val"
PROCESSED_DIR = DATA_ROOT / "processed"

# Dlib Method D Cropping Parameters
TARGET_WIDTH = 96
TARGET_HEIGHT = 96
PADDING_X = 15      # Horizontal margin in px
PADDING_Y = 10      # Vertical margin in px
BORDER_VALUE = 0    # Black letterbox canvas fill
JPEG_QUALITY = 95

# Dataset Target Split Counts
TARGET_TRAIN_FOLDERS = 80
TARGET_VAL_FOLDERS = 20
TOTAL_TARGET_FOLDERS = TARGET_TRAIN_FOLDERS + TARGET_VAL_FOLDERS

print(f"Project Root:             {PROJECT_ROOT}")
print(f"Raw Digit Frames:         {RAW_DIGIT_DIR}")
print(f"Pre-cropped Method D Dir: {PRE_CROPPED_METHOD_D_DIR}")
print(f"Dlib Landmark Predictor:  {DLIB_PREDICTOR_PATH}")
print(f"Output Dataset Root:      {OUTPUT_DATASET_DIR}")
print(f"Output Train Directory:   {OUTPUT_TRAIN_DIR} (Target: {TARGET_TRAIN_FOLDERS} folders)")
print(f"Output Val Directory:     {OUTPUT_VAL_DIR} (Target: {TARGET_VAL_FOLDERS} folders)")

## 2. Zero-Leakage 80/20 Base-Speaker Partition & 100-Folder Expansion

We enforce strict speaker independence with **zero data leakage**:
- **24 Base Speakers for Train**: `s1` through `s24` (80% of 30 speakers)
- **6 Base Speakers for Validation**: `s25` through `s30` (20% of 30 speakers)

To expand 24 base speakers into exactly 80 train directories and 6 base speakers into exactly 20 validation directories:
- **Train (24 base $\to$ 80 folders)**:
  - Duplication quotient: $80 // 24 = 3$ base directories per speaker (1 original + 2 augmentations).
  - Remainder: $80 \% 24 = 8$ speakers receive 1 extra augmentation (1 original + 3 augmentations).
  - Allocation: $8 \times 4 + 16 \times 3 = 32 + 48 = 80$ folders.
- **Validation (6 base $\to$ 20 folders)**:
  - Duplication quotient: $20 // 6 = 3$ base directories per speaker (1 original + 2 augmentations).
  - Remainder: $20 \% 6 = 2$ speakers receive 1 extra augmentation (1 original + 3 augmentations).
  - Allocation: $2 \times 4 + 4 \times 3 = 8 + 12 = 20$ folders.

In [ ]:
def natural_sort_key(s: str) -> int:
    m = re.search(r"\d+", s)
    return int(m.group(0)) if m else 0

# Canonical 30 Base Speakers
ALL_BASE_SPEAKERS = sorted([f"s{i}" for i in range(1, 31)], key=natural_sort_key)

# 80/20 Partition: 24 Train base speakers, 6 Validation base speakers
TRAIN_BASE_SPEAKERS = sorted([f"s{i}" for i in range(1, 25)], key=natural_sort_key)
VAL_BASE_SPEAKERS = sorted([f"s{i}" for i in range(25, 31)], key=natural_sort_key)

def build_speaker_folder_mapping(
    base_speakers: List[str],
    target_count: int,
    prefix_aug: str = "aug"
) -> Dict[str, List[str]]:
    """
    Deterministically maps base speakers to exact folder directories (original + augmentations)
    to achieve the exact target folder count.
    """
    n_base = len(base_speakers)
    assert target_count >= n_base, f"Target count {target_count} < base speakers {n_base}"
    
    extra_folders_needed = target_count - n_base
    base_augs_per_spk = extra_folders_needed // n_base
    remainder = extra_folders_needed % n_base
    
    mapping: Dict[str, List[str]] = {}
    for idx, spk in enumerate(base_speakers):
        dirs = [spk]  # original crop folder
        num_augs = base_augs_per_spk + (1 if idx < remainder else 0)
        for a_idx in range(1, num_augs + 1):
            dirs.append(f"{spk}_{prefix_aug}{a_idx}")
        mapping[spk] = dirs
        
    total_dirs = sum(len(v) for v in mapping.values())
    assert total_dirs == target_count, f"Generated {total_dirs} != target {target_count}"
    return mapping

# Generate Folder Mappings
train_mapping = build_speaker_folder_mapping(TRAIN_BASE_SPEAKERS, TARGET_TRAIN_FOLDERS)
val_mapping = build_speaker_folder_mapping(VAL_BASE_SPEAKERS, TARGET_VAL_FOLDERS)

# Flatten directory lists
all_train_folders = sorted([folder for dirs in train_mapping.values() for folder in dirs], key=natural_sort_key)
all_val_folders = sorted([folder for dirs in val_mapping.values() for folder in dirs], key=natural_sort_key)

# Strict Zero-Leakage & Consistency Assertions
assert len(TRAIN_BASE_SPEAKERS) == 24, "Train base speakers must be exactly 24"
assert len(VAL_BASE_SPEAKERS) == 6, "Val base speakers must be exactly 6"
assert len(set(TRAIN_BASE_SPEAKERS) & set(VAL_BASE_SPEAKERS)) == 0, "Zero base speaker overlap"
assert len(all_train_folders) == 80, f"Expected 80 train folders, got {len(all_train_folders)}"
assert len(all_val_folders) == 20, f"Expected 20 val folders, got {len(all_val_folders)}"
assert len(set(all_train_folders) & set(all_val_folders)) == 0, "Zero folder overlap between train and val"
assert len(all_train_folders) + len(all_val_folders) == 100, "Total folders must equal 100"

print("=" * 80)
print("      ZERO-LEAKAGE 80/20 BASE SPEAKER & FOLDER ALLOCATION VERIFIED")
print("=" * 80)
print(f"Base Train Speakers ({len(TRAIN_BASE_SPEAKERS)}): {TRAIN_BASE_SPEAKERS}")
print(f"Base Val Speakers   ({len(VAL_BASE_SPEAKERS)}): {VAL_BASE_SPEAKERS}")
print("-" * 80)
print(f"Train Directories ({len(all_train_folders)}):")
print(f"  - 8 base speakers with 4 folders (1 orig + 3 augs): {TRAIN_BASE_SPEAKERS[:8]}")
print(f"  - 16 base speakers with 3 folders (1 orig + 2 augs): {TRAIN_BASE_SPEAKERS[8:]}")
print(f"Validation Directories ({len(all_val_folders)}):")
print(f"  - 2 base speakers with 4 folders (1 orig + 3 augs): {VAL_BASE_SPEAKERS[:2]}")
print(f"  - 4 base speakers with 3 folders (1 orig + 2 augs): {VAL_BASE_SPEAKERS[2:]}")
print(f"Total Folders: {len(all_train_folders) + len(all_val_folders)} (80 Train + 20 Val)")
print("=" * 80)

## 3. Dlib Landmark Extraction & Method D Letterbox Cropping

Method D letterboxing extracts the mouth region tightly and embeds it into a $96 \times 96$ square canvas without vertical stretching:
1. Mouth landmarks 48–67 define raw mouth coordinates.
2. Centroid $(c_x, c_y)$ and bounding box $(w_{\text{raw}}, h_{\text{raw}})$ are computed.
3. Margins $P_x = 15\text{ px}$ and $P_y = 10\text{ px}$ are added.
4. Aspect-preserving scale is computed: $\text{scale} = 96 / \max(W_{\text{roi}}, H_{\text{roi}})$.
5. Scaled mouth is centered on a neutral black canvas of size $96 \times 96 \times 3$.

*Note*: If `data/experiments/cropped_lips_dataset_d_96x96` or `results/dlib/dlib_roi_geometry_all_frames.csv` is already present, we can read base crops directly or utilize pre-computed geometry for maximum performance, with live Dlib detection available as a standalone fallback.

In [ ]:
def extract_method_d_crop(
    img: np.ndarray,
    center_x: float,
    center_y: float,
    raw_width: float,
    raw_height: float,
    pad_x: int = PADDING_X,
    pad_y: int = PADDING_Y,
    canvas_size: int = TARGET_WIDTH,
    border_val: int = BORDER_VALUE
) -> Tuple[np.ndarray, int, int, int, int]:
    """
    Extracts an aspect-preserving letterboxed mouth ROI from full frame image:
    1. Bounding box computation with padding
    2. Proportional aspect-preserving scaling
    3. Neutral centered letterboxing on canvas_size x canvas_size
    """
    h_img, w_img = img.shape[:2]
    
    min_x = center_x - raw_width / 2.0
    max_x = center_x + raw_width / 2.0
    min_y = center_y - raw_height / 2.0
    max_y = center_y + raw_height / 2.0
    
    x1 = max(0, int(round(min_x - pad_x)))
    y1 = max(0, int(round(min_y - pad_y)))
    x2 = min(w_img, int(round(max_x + pad_x)))
    y2 = min(h_img, int(round(max_y + pad_y)))
    
    roi = img[y1:y2, x1:x2]
    h_roi, w_roi = roi.shape[:2]
    
    if h_roi == 0 or w_roi == 0:
        return np.full((canvas_size, canvas_size, 3), border_val, dtype=np.uint8), 0, 0, 0, 0
        
    scale = canvas_size / max(w_roi, h_roi)
    new_w = min(canvas_size, max(1, int(round(w_roi * scale))))
    new_h = min(canvas_size, max(1, int(round(h_roi * scale))))
    
    resized_roi = cv2.resize(roi, (new_w, new_h), interpolation=cv2.INTER_AREA)
    
    canvas = np.full((canvas_size, canvas_size, 3), border_val, dtype=np.uint8)
    offset_x = (canvas_size - new_w) // 2
    offset_y = (canvas_size - new_h) // 2
    
    canvas[offset_y:offset_y + new_h, offset_x:offset_x + new_w] = resized_roi
    pad_h = canvas_size - new_w
    pad_v = canvas_size - new_h
    
    return canvas, new_w, new_h, pad_h, pad_v

class DlibLipDetector:
    """
    Wrapper around Dlib 68-point landmark predictor for direct mouth extraction.
    """
    def __init__(self, predictor_path: Path):
        self.detector = dlib.get_frontal_face_detector()
        self.predictor = dlib.shape_predictor(str(predictor_path)) if predictor_path.exists() else None

    def get_mouth_geometry(self, img_bgr: np.ndarray) -> Optional[Dict[str, float]]:
        if self.predictor is None:
            return None
        gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        faces = self.detector(gray, 0)
        if not faces:
            return None
        face = faces[0]
        shape = self.predictor(gray, face)
        # Landmarks 48-67 represent the mouth
        pts = np.array([[shape.part(i).x, shape.part(i).y] for i in range(48, 68)], dtype=np.float32)
        min_x, min_y = pts.min(axis=0)
        max_x, max_y = pts.max(axis=0)
        raw_w = float(max_x - min_x)
        raw_h = float(max_y - min_y)
        cx = float((min_x + max_x) / 2.0)
        cy = float((min_y + max_y) / 2.0)
        return {
            "center_x": cx,
            "center_y": cy,
            "raw_width": raw_w,
            "raw_height": raw_h
        }

print("Method D letterbox cropping functions initialized.")

## 4. Lip-Reading Safe Augmentation Transformations

Augmentations are designed specifically for Visual Speech Recognition (VSR):
1. **Geometric Consistency**: Transformations must be temporally consistent across all frames of an utterance to avoid artificial jitter. We sample affine parameters $(\Delta x, \Delta y, \theta, s)$ deterministically per utterance variant using SHA-256 seeding.
2. **Photometric Realism**: Contrast, brightness, subtle sensor noise, and slight blur vary per frame to simulate real lighting and camera fluctuations.
3. **Boundary Integrity**: `cv2.BORDER_REFLECT_101` eliminates unnatural black border artifacts when shifting letterboxed frames.

In [ ]:
def get_geometric_rng(base_seed: int, speaker: str, digit: str, variant_idx: int) -> np.random.Generator:
    """
    Deterministic RNG per (base_seed, speaker, digit, variant) to preserve
    identical geometric alignment across all frames of an utterance.
    """
    key = f"{base_seed}_{speaker}_{digit}_aug{variant_idx}_geom".encode("utf-8")
    seed_uint32 = int.from_bytes(hashlib.sha256(key).digest()[:4], byteorder="little")
    return np.random.default_rng(seed_uint32)

def get_photometric_rng(base_seed: int, speaker: str, digit: str, frame: str, variant_idx: int) -> np.random.Generator:
    """
    Deterministic RNG per frame for photometric variations.
    """
    key = f"{base_seed}_{speaker}_{digit}_{frame}_aug{variant_idx}_photo".encode("utf-8")
    seed_uint32 = int.from_bytes(hashlib.sha256(key).digest()[:4], byteorder="little")
    return np.random.default_rng(seed_uint32)

def sample_geometric_transform(
    rng: np.random.Generator,
    width: int = TARGET_WIDTH,
    height: int = TARGET_HEIGHT
) -> np.ndarray:
    """
    Conservative affine transformation parameters:
    - Translation: +/- 3.0 px
    - Rotation: +/- 3.0 degrees
    - Scale: 0.97 to 1.03
    """
    tx = float(rng.uniform(-3.0, 3.0))
    ty = float(rng.uniform(-3.0, 3.0))
    angle = float(rng.uniform(-3.0, 3.0))
    scale = float(rng.uniform(0.97, 1.03))
    
    center = (width / 2.0, height / 2.0)
    M = cv2.getRotationMatrix2D(center, angle, scale)
    M[0, 2] += tx
    M[1, 2] += ty
    return M

def apply_conservative_augmentation(
    img: np.ndarray,
    M: np.ndarray,
    rng: np.random.Generator
) -> np.ndarray:
    """
    Applies geometric affine warp and photometric adjustments.
    Guarantees strict 96x96x3 output shape.
    """
    h, w = img.shape[:2]
    
    # 1. Geometric warp with reflection border to avoid synthetic edge cuts
    transformed = cv2.warpAffine(
        img,
        M,
        (w, h),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_REFLECT_101
    )
    
    # 2. Photometric: Contrast (0.92 - 1.08) and Brightness (-10 to +10)
    alpha = float(rng.uniform(0.92, 1.08))
    beta = float(rng.uniform(-10.0, 10.0))
    photo = np.clip(alpha * transformed.astype(np.float32) + beta, 0.0, 255.0).astype(np.uint8)
    
    # 3. Gaussian sensor noise (50% probability, sigma 1.0 - 3.5)
    if rng.random() < 0.50:
        sigma = float(rng.uniform(1.0, 3.5))
        noise = rng.normal(0.0, sigma, photo.shape)
        photo = np.clip(photo.astype(np.float32) + noise, 0.0, 255.0).astype(np.uint8)
        
    # 4. Mild Gaussian blur (30% probability, 3x3 kernel, sigma 0.5)
    if rng.random() < 0.30:
        photo = cv2.GaussianBlur(photo, (3, 3), 0.5)
        
    return photo

print("Augmentation transformation functions validated.")

## 5. End-to-End Dataset Generation Pipeline

This module executes the crop & augmentation engine:
1. Discovers input frames (using pre-cropped Method D dataset when available for rapid processing, or cropping directly via geometry index/Dlib).
2. For each base speaker, extracts and copies the unaugmented base frames to the output folder.
3. For each assigned augmentation variant (`_aug1`, `_aug2`, `_aug3`), generates the corresponding variant frames with sequence-level geometric consistency.
4. Distributes outputs cleanly into:
   - `data/experiments/dlib_augmented_cropped_100_96x96/train/` (80 folders)
   - `data/experiments/dlib_augmented_cropped_100_96x96/val/` (20 folders)
5. Generates a comprehensive CSV manifest (`augmentation_manifest.csv`) and JSON summary.

In [ ]:
def collect_base_speaker_frames(base_speaker: str) -> List[Tuple[str, str, Path]]:
    """
    Collects all (digit, filename, source_path) for a base speaker.
    Checks pre-cropped Method D directory first; if not found, falls back to raw digit directory.
    """
    frames = []
    
    # Check pre-cropped Method D dataset first
    spk_cropped_dir = PRE_CROPPED_METHOD_D_DIR / base_speaker
    if spk_cropped_dir.exists():
        for dig_dir in sorted(spk_cropped_dir.iterdir()):
            if dig_dir.is_dir() and not dig_dir.name.startswith("."):
                dig = dig_dir.name
                for fpath in sorted(dig_dir.glob("*.jpg")):
                    frames.append((dig, fpath.name, fpath))
        if frames:
            return frames
            
    # Fallback: check raw digit directory
    spk_raw_dir = RAW_DIGIT_DIR / base_speaker
    if spk_raw_dir.exists():
        for dig_dir in sorted(spk_raw_dir.iterdir()):
            if dig_dir.is_dir() and not dig_dir.name.startswith("."):
                dig = dig_dir.name
                for fpath in sorted(dig_dir.glob("*.jpg")):
                    frames.append((dig, fpath.name, fpath))
    return frames

def process_speaker_partition(
    base_speaker: str,
    target_folders: List[str],
    split_name: str,
    output_split_dir: Path,
    base_seed: int = DEFAULT_SEED,
    geometry_dict: Optional[Dict[str, Any]] = None,
    overwrite: bool = False
) -> List[Dict[str, Any]]:
    """
    Processes one base speaker:
    - Writes original folder (if in target_folders)
    - Writes each augmented variant folder
    """
    records = []
    frame_list = collect_base_speaker_frames(base_speaker)
    if not frame_list:
        print(f"Warning: No frames found for speaker {base_speaker}")
        return records

    unique_digits = sorted({dig for dig, _, _ in frame_list})
    
    # Identify which augmented variants need to be generated
    # e.g., target_folders might be ["s1", "s1_aug1", "s1_aug2", "s1_aug3"]
    aug_variants = [f for f in target_folders if f != base_speaker]
    
    # Pre-sample geometric transforms once per (digit, variant)
    geo_transforms: Dict[Tuple[str, str], np.ndarray] = {}
    for dig in unique_digits:
        for v_folder in aug_variants:
            m = re.search(r"aug(\d+)", v_folder)
            v_idx = int(m.group(1)) if m else 1
            geo_rng = get_geometric_rng(base_seed, base_speaker, dig, v_idx)
            geo_transforms[(dig, v_folder)] = sample_geometric_transform(geo_rng, TARGET_WIDTH, TARGET_HEIGHT)

    # Process all frames
    for dig, fname, src_path in frame_list:
        # 1. Base unaugmented frame
        if base_speaker in target_folders:
            dst_orig_path = output_split_dir / base_speaker / dig / fname
            dst_orig_path.parent.mkdir(parents=True, exist_ok=True)
            if overwrite or not dst_orig_path.exists():
                shutil.copy2(src_path, dst_orig_path)
            records.append({
                "split": split_name,
                "base_speaker": base_speaker,
                "folder_name": base_speaker,
                "is_augmented": False,
                "variant": "original",
                "digit": dig,
                "frame": fname,
                "source_path": str(src_path),
                "output_path": str(dst_orig_path),
                "width": TARGET_WIDTH,
                "height": TARGET_HEIGHT
            })
            
        # Read image once for augmentations
        img = cv2.imread(str(src_path))
        if img is None:
            continue
            
        # Ensure base image is 96x96 letterbox
        if img.shape[0] != TARGET_HEIGHT or img.shape[1] != TARGET_WIDTH:
            key = f"{base_speaker}/{dig}/{fname}"
            if geometry_dict and key in geometry_dict:
                geom = geometry_dict[key]
                img, _, _, _, _ = extract_method_d_crop(
                    img, geom["center_x"], geom["center_y"], geom["raw_width"], geom["raw_height"]
                )
            else:
                img = cv2.resize(img, (TARGET_WIDTH, TARGET_HEIGHT))

        # 2. Augmented variants
        for v_folder in aug_variants:
            m = re.search(r"aug(\d+)", v_folder)
            v_idx = int(m.group(1)) if m else 1
            dst_aug_path = output_split_dir / v_folder / dig / fname
            dst_aug_path.parent.mkdir(parents=True, exist_ok=True)
            
            if overwrite or not dst_aug_path.exists():
                M = geo_transforms[(dig, v_folder)]
                photo_rng = get_photometric_rng(base_seed, base_speaker, dig, fname, v_idx)
                aug_img = apply_conservative_augmentation(img, M, photo_rng)
                cv2.imwrite(str(dst_aug_path), aug_img, [int(cv2.IMWRITE_JPEG_QUALITY), JPEG_QUALITY])
                
            records.append({
                "split": split_name,
                "base_speaker": base_speaker,
                "folder_name": v_folder,
                "is_augmented": True,
                "variant": f"aug{v_idx}",
                "digit": dig,
                "frame": fname,
                "source_path": str(src_path),
                "output_path": str(dst_aug_path),
                "width": TARGET_WIDTH,
                "height": TARGET_HEIGHT
            })
            
    return records

print("Dataset processing worker function defined.")

## 6. Execution, Manifest Generation & Zero-Leakage Verification

We execute the generation pipeline across both splits:
- **Train partition**: 24 base speakers $\to$ 80 folders
- **Validation partition**: 6 base speakers $\to$ 20 folders
- Export master manifest (`augmentation_manifest.csv`) and `speaker_split.json`.
- Execute strict automated verification checks.

In [ ]:
t0 = time.time()
OUTPUT_TRAIN_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_VAL_DIR.mkdir(parents=True, exist_ok=True)

# Load geometry dictionary if available
geom_dict = None
if GEOMETRY_CSV_PATH.exists():
    df_geom = pd.read_csv(GEOMETRY_CSV_PATH)
    geom_dict = {
        f"{r['speaker']}/{r['digit']}/{r['frame']}": {
            "center_x": r["center_x"], "center_y": r["center_y"],
            "raw_width": r["raw_width"], "raw_height": r["raw_height"]
        }
        for _, r in df_geom.iterrows()
    }
    print(f"Loaded {len(geom_dict):,} landmark geometry records from {GEOMETRY_CSV_PATH.name}")

all_manifest_records = []

# 1. Generate Train Split (80 Folders)
print("\nProcessing Train Split (24 base speakers -> 80 folders)...")
for spk in tqdm(TRAIN_BASE_SPEAKERS, desc="Train Speakers"):
    target_dirs = train_mapping[spk]
    rec = process_speaker_partition(
        base_speaker=spk,
        target_folders=target_dirs,
        split_name="train",
        output_split_dir=OUTPUT_TRAIN_DIR,
        base_seed=DEFAULT_SEED,
        geometry_dict=geom_dict,
        overwrite=False
    )
    all_manifest_records.extend(rec)

# 2. Generate Validation Split (20 Folders)
print("\nProcessing Validation Split (6 base speakers -> 20 folders)...")
for spk in tqdm(VAL_BASE_SPEAKERS, desc="Validation Speakers"):
    target_dirs = val_mapping[spk]
    rec = process_speaker_partition(
        base_speaker=spk,
        target_folders=target_dirs,
        split_name="val",
        output_split_dir=OUTPUT_VAL_DIR,
        base_seed=DEFAULT_SEED,
        geometry_dict=geom_dict,
        overwrite=False
    )
    all_manifest_records.extend(rec)

elapsed_sec = time.time() - t0
print(f"\nProcessing completed in {elapsed_sec:.2f} seconds!")

# 3. Export Manifests & Summary Metadata
df_manifest = pd.DataFrame(all_manifest_records)
manifest_path = OUTPUT_DATASET_DIR / "augmentation_manifest.csv"
df_manifest.to_csv(manifest_path, index=False)
print(f"Saved manifest ({len(df_manifest):,} records) to: {manifest_path}")

# Export Speaker Split JSON
split_json_data = {
    "dataset_name": DATASET_NAME,
    "seed": DEFAULT_SEED,
    "num_train_base_speakers": len(TRAIN_BASE_SPEAKERS),
    "num_val_base_speakers": len(VAL_BASE_SPEAKERS),
    "train_base_speakers": TRAIN_BASE_SPEAKERS,
    "val_base_speakers": VAL_BASE_SPEAKERS,
    "target_train_folders": TARGET_TRAIN_FOLDERS,
    "target_val_folders": TARGET_VAL_FOLDERS,
    "total_folders": TOTAL_TARGET_FOLDERS,
    "train_folder_mapping": train_mapping,
    "val_folder_mapping": val_mapping,
    "train_folders": all_train_folders,
    "val_folders": all_val_folders,
    "zero_leakage_verified": True
}
split_json_path = OUTPUT_DATASET_DIR / "speaker_split.json"
with open(split_json_path, "w") as f:
    json.dump(split_json_data, f, indent=2)
print(f"Saved split metadata to: {split_json_path}")

# Export Summary JSON
summary_data = {
    "dataset_name": DATASET_NAME,
    "total_folders": len(all_train_folders) + len(all_val_folders),
    "train_folders_count": len(all_train_folders),
    "val_folders_count": len(all_val_folders),
    "train_frames_count": int((df_manifest["split"] == "train").sum()),
    "val_frames_count": int((df_manifest["split"] == "val").sum()),
    "total_frames_count": len(df_manifest),
    "dimensions": [TARGET_WIDTH, TARGET_HEIGHT, 3],
    "elapsed_seconds": round(elapsed_sec, 2)
}
summary_json_path = OUTPUT_DATASET_DIR / "augmentation_summary.json"
with open(summary_json_path, "w") as f:
    json.dump(summary_data, f, indent=2)
print(f"Saved summary to: {summary_json_path}")

## 7. Strict Integrity & Zero-Leakage Assertions

We execute comprehensive verification tests ensuring the generated dataset adheres strictly to all requirements:
1. **Directory Count Check**: Exactly 80 train directories and 20 validation directories (100 total).
2. **Zero-Leakage Check**: No speaker or directory overlap between train and validation.
3. **Digit Completeness**: Every directory contains all 11 digits (`d0` through `d10`).
4. **Dimension Verification**: Output frames are verified to be exact $96 \times 96 \times 3$.

In [ ]:
# 1. Count Directories on Disk
train_disk_dirs = sorted([d.name for d in OUTPUT_TRAIN_DIR.iterdir() if d.is_dir() and not d.name.startswith(".")], key=natural_sort_key)
val_disk_dirs = sorted([d.name for d in OUTPUT_VAL_DIR.iterdir() if d.is_dir() and not d.name.startswith(".")], key=natural_sort_key)

print("=" * 80)
print("             DATASET INTEGRITY & ZERO-LEAKAGE VERIFICATION")
print("=" * 80)
print(f"Train Directories on Disk:      {len(train_disk_dirs)} (Expected: {TARGET_TRAIN_FOLDERS})")
print(f"Validation Directories on Disk: {len(val_disk_dirs)} (Expected: {TARGET_VAL_FOLDERS})")
print(f"Total Directories on Disk:      {len(train_disk_dirs) + len(val_disk_dirs)} (Expected: {TOTAL_TARGET_FOLDERS})")
print("-" * 80)

# Check 1: Exact directory counts
assert len(train_disk_dirs) == TARGET_TRAIN_FOLDERS, f"Train folder mismatch: {len(train_disk_dirs)} != {TARGET_TRAIN_FOLDERS}"
assert len(val_disk_dirs) == TARGET_VAL_FOLDERS, f"Val folder mismatch: {len(val_disk_dirs)} != {TARGET_VAL_FOLDERS}"
print("[PASS] Directory counts match target (80 Train + 20 Val = 100 Folders).")

# Check 2: Zero Leakage
overlap = set(train_disk_dirs) & set(val_disk_dirs)
assert len(overlap) == 0, f"Data leakage detected! Overlapping folders: {overlap}"
base_train_in_val = set([d.split("_")[0] for d in train_disk_dirs]) & set([d.split("_")[0] for d in val_disk_dirs])
assert len(base_train_in_val) == 0, f"Base speaker leakage detected: {base_train_in_val}"
print("[PASS] Zero data leakage: Train and Validation partitions are strictly disjoint.")

# Check 3: Digit Completeness across all 100 folders
for d_name in train_disk_dirs:
    p = OUTPUT_TRAIN_DIR / d_name
    digits = [sub.name for sub in p.iterdir() if sub.is_dir() and not sub.name.startswith(".")]
    assert len(digits) == 11, f"Folder {d_name} has {len(digits)} digits, expected 11"

for d_name in val_disk_dirs:
    p = OUTPUT_VAL_DIR / d_name
    digits = [sub.name for sub in p.iterdir() if sub.is_dir() and not sub.name.startswith(".")]
    assert len(digits) == 11, f"Folder {d_name} has {len(digits)} digits, expected 11"
print("[PASS] Utterance completeness: All 100 directories contain all 11 digits (d0..d10).")

# Check 4: Frame Dimensions
sample_frames = list(OUTPUT_TRAIN_DIR.glob("*/*/*.jpg"))[:50] + list(OUTPUT_VAL_DIR.glob("*/*/*.jpg"))[:50]
for sf in sample_frames:
    im = cv2.imread(str(sf))
    assert im is not None, f"Unreadable frame: {sf}"
    assert im.shape == (TARGET_HEIGHT, TARGET_WIDTH, 3), f"Invalid frame dimensions: {im.shape} in {sf}"
print(f"[PASS] Dimension integrity: Verified sample frames are exactly {TARGET_WIDTH}x{TARGET_HEIGHT}x3 RGB.")
print("=" * 80)
print("                 ALL INTEGRITY CHECKS PASSED (100%)")
print("=" * 80)

## 8. Visual Inspection & Quality Analysis

Inspect side-by-side visual samples comparing:
- Original letterboxed lip crop ($96 \times 96$)
- Augmented variants (`_aug1`, `_aug2`, `_aug3`)
- Visualizing both Train and Validation speakers

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(12, 12))

# Select sample speakers and digits
sample_cases = [
    ("s1", "d0", "Train (s1 Base)"),
    ("s8", "d5", "Train (s8 Base)"),
    ("s25", "d0", "Val (s25 Base)"),
    ("s30", "d9", "Val (s30 Base)"),
]

for row_idx, (spk, dig, label) in enumerate(sample_cases):
    split_dir = OUTPUT_TRAIN_DIR if spk in TRAIN_BASE_SPEAKERS else OUTPUT_VAL_DIR
    variants = [spk, f"{spk}_aug1", f"{spk}_aug2", f"{spk}_aug3"]
    
    for col_idx, var in enumerate(variants):
        var_dir = split_dir / var / dig
        ax = axes[row_idx, col_idx]
        
        if var_dir.exists():
            fpaths = sorted(var_dir.glob("*.jpg"))
            if fpaths:
                mid_frame = fpaths[len(fpaths) // 2]
                im_bgr = cv2.imread(str(mid_frame))
                im_rgb = cv2.cvtColor(im_bgr, cv2.COLOR_BGR2RGB)
                ax.imshow(im_rgb)
                ax.set_title(f"{var} ({dig})\n{mid_frame.name}", fontsize=9)
            else:
                ax.text(0.5, 0.5, "No frames", ha="center", va="center")
        else:
            ax.text(0.5, 0.5, "N/A", ha="center", va="center")
            
        ax.axis("off")
    axes[row_idx, 0].set_ylabel(label, fontsize=10, fontweight="bold")

plt.suptitle("Method D (Letterboxed 96x96): Original vs Conservative Augmented Variants", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 9. Active Dataset Linker (Optional Switch to `data/processed`)

To immediately train baseline models (`05_...`, `06_...`, `11_...`, etc.) on this newly generated 100-folder dataset, run the cell below. It updates `data/processed/train` (80 folders) and `data/processed/val` (20 folders) using zero-leakage symlinks.

In [ ]:
def switch_active_processed_dataset(
    source_train_dir: Path = OUTPUT_TRAIN_DIR,
    source_val_dir: Path = OUTPUT_VAL_DIR,
    processed_dir: Path = PROCESSED_DIR
) -> None:
    """
    Links data/processed/train to source_train_dir (80 folders)
    and data/processed/val to source_val_dir (20 folders).
    """
    train_dest = processed_dir / "train"
    val_dest = processed_dir / "val"
    test_dest = processed_dir / "test"
    
    def clear_and_relink(dest_dir: Path, src_dir: Path):
        dest_dir.mkdir(parents=True, exist_ok=True)
        # Clear existing contents
        for item in dest_dir.iterdir():
            if item.name.startswith("."):
                continue
            if item.is_symlink() or item.is_file():
                item.unlink()
            elif item.is_dir():
                shutil.rmtree(item)
                
        # Link all subfolders
        count = 0
        for sub_dir in sorted(src_dir.iterdir()):
            if sub_dir.is_dir() and not sub_dir.name.startswith("."):
                target_symlink = dest_dir / sub_dir.name
                target_symlink.symlink_to(sub_dir.resolve())
                count += 1
        return count

    n_train = clear_and_relink(train_dest, source_train_dir)
    n_val = clear_and_relink(val_dest, source_val_dir)
    
    # Ensure test directory is clean
    test_dest.mkdir(parents=True, exist_ok=True)
    for item in test_dest.iterdir():
        if not item.name.startswith("."):
            if item.is_symlink() or item.is_file():
                item.unlink()
            elif item.is_dir():
                shutil.rmtree(item)
                
    print("=" * 80)
    print("        ACTIVE DATASET SWITCHED TO 100-FOLDER METHOD D DATASET")
    print("=" * 80)
    print(f"data/processed/train: {n_train} linked folders (Expected: 80)")
    print(f"data/processed/val:   {n_val} linked folders (Expected: 20)")
    print(f"data/processed/test:  0 linked folders (empty for 80/20 train/val protocol)")
    print("=" * 80)

# Uncomment to activate this dataset in data/processed:
# switch_active_processed_dataset()
print("Run `switch_active_processed_dataset()` to activate in data/processed.")

### Summary of Generated Dataset:
| Specification | Train Split | Validation Split | Total |
| :--- | :--- | :--- | :--- |
| **Base Speakers** | 24 (`s1`..`s24`) | 6 (`s25`..`s30`) | **30 base speakers** |
| **Folder Expansion** | 8 spk $\times$ 4 + 16 spk $\times$ 3 | 2 spk $\times$ 4 + 4 spk $\times$ 3 | **100 speaker folders** |
| **Speaker Directories** | **80 folders** | **20 folders** | **100 folders** |
| **Digits per Folder** | 11 (`d0`..`d10`) | 11 (`d0`..`d10`) | 11 digits |
| **Lip Crop Method** | Method D (Letterboxed) | Method D (Letterboxed) | Method D (Letterboxed) |
| **Crop Dimensions** | $96 \times 96 \times 3$ | $96 \times 96 \times 3$ | $96 \times 96 \times 3$ |
| **Data Leakage** | **0% (Strictly Disjoint)** | **0% (Strictly Disjoint)** | **Zero Data Leakage** |